# FlashInspector AI – Test Model on Video + Feedback Loop

Upload a video and run your trained YOLOv8 models on it, then **correct mistakes and retrain**.

Runs **both** models on every frame and combines the annotations:
- **Detection** (`best_detect.pt`) — bounding boxes for fire safety equipment (S1,2,5,6)
- **Segmentation** (`best_segment.pt`) — masks for extinguisher condition/occlusion (S3,4)

### Violation detection (from inspection video analysis)
| Violation | How it's detected | Class(es) needed |
|-----------|-------------------|-----------------|
| **Extinguisher missing** | `empty_mount` detected with no nearby extinguisher | `empty_mount`, `extinguisher_cabinet_empty` |
| **Non-compliant tag** | Red/yellow tag on extinguisher | `non_compliant_tag` |
| **Extinguisher out of date** | OCR reads expiry date from tag/label (S5) | existing S5 classes |
| **Exit sign not illuminated** | Dark/unlit exit sign detected | `exit_sign_dark` |
| **Smoke detector missing from base** | Ceiling base without detector | `smoke_detector_missing` |
| **Notification appliance too high** | Appliance detected + position check | `notification_appliance` |
| **Pull station too high** | Pull station detected + position check | `pull_station` |
| **Fire alarm panel trouble** | Panel detected + LED color analysis | `fire_alarm_panel` |
| **Exit blocked** | Blocked exit detected | `blocked_exit` |

### Feedback loop (human-in-the-loop retraining)
After inference, you can **review detections frame-by-frame**, correct mistakes
(remove false positives, reclassify, add missed objects), and **fine-tune the model**
on your corrections. Run inference again to see the improvement — repeat until satisfied.

```
Infer → Review → Correct → Retrain → Re-infer → ...
```

## 1. Check GPU

Go to **Runtime → Change runtime type → T4 GPU** if no GPU is detected.

In [ ]:
!nvidia-smi 2>/dev/null || echo "\n\u26a0\ufe0f No GPU detected. Go to Runtime \u2192 Change runtime type \u2192 T4 GPU."

## 2. Install dependencies & clone repo

In [ ]:
!pip install -q ultralytics opencv-python easyocr

import os
from pathlib import Path
from google.colab import userdata

# Reset to safe directory first (in case previous dir was deleted)
try:
    os.getcwd()
except:
    os.chdir('/content')

# --- Private repo: use a GitHub Personal Access Token ---
token = userdata.get('GITHUB_TOKEN')

repo_dir = Path("/content/fire")
project_dir = repo_dir / "flashinspector-ai"

# Clone if missing, otherwise pull latest changes
if not project_dir.exists():
    os.chdir('/content')
    if repo_dir.exists():
        !rm -rf /content/fire
    !git clone https://{token}@github.com/patrisiyarum/fire.git /content/fire
else:
    os.chdir(str(repo_dir))
    !git pull origin main --ff-only 2>/dev/null || echo "Pull skipped (local changes exist)"
    print("Repo updated.")

os.chdir(str(project_dir))
print("\nProject ready at:", Path.cwd())

## 3. Load models from repo

Loads models directly from the cloned repository:
- **`best_detect.pt`** — detection (S1: equipment, S2: marking signs, S5: inspection tags, S6: fire class symbols)
- **`best_segment.pt`** — segmentation (S3/S4: extinguisher condition/occlusion)
- **`best.pt`** — fallback detection model

**Violation classes** (train via feedback loop):
- `empty_mount` / `extinguisher_cabinet_empty` → **EXTINGUISHER MISSING**
- `non_compliant_tag` → **NON-COMPLIANT** (red/yellow tag on extinguisher)
- `exit_sign_dark` → **EXIT SIGN NOT ILLUMINATED**
- `smoke_detector_missing` → **SMOKE DETECTOR MISSING**
- `notification_appliance` → mounting height check
- `pull_station` → mounting height check
- `fire_alarm_panel` → panel status check
- `blocked_exit` → **EXIT BLOCKED**

In [ ]:
from pathlib import Path
from ultralytics import YOLO

# --- Service classification for model classes ---
SERVICE_KEYWORDS = {
    "S5-inspection_tag": ["tag", "inspection", "expir", "date", "maintenance", "servic"],
    "S6-fire_class": ["class", "symbol", "type_a", "type_b", "type_c", "type_d", "type_e", "type_f",
                       "class_a", "class_b", "class_c", "class_d", "class_e", "class_f",
                       "powder", "foam", "co2", "water", "wet_chemical"],
    "S2-marking": ["marking", "sign", "signage", "suppression"],
    "S1-equipment": ["extinguisher", "blanket", "detector", "call_point", "alarm", "sounder",
                     "exit", "dome", "flash", "light", "orb", "hydrant", "emergency_light"],
    "VIOLATION-missing": ["empty_mount", "cabinet_empty", "bracket_empty", "missing_extinguisher"],
    "VIOLATION-noncompliant": ["non_compliant", "noncompliant", "yellow_tag", "red_tag"],
    "VIOLATION-exit_dark": ["exit_sign_dark", "exit_dark", "sign_dark", "unlit_exit"],
    "VIOLATION-smoke_missing": ["smoke_detector_missing", "detector_missing", "detector_base"],
    "VIOLATION-blocked_exit": ["blocked_exit", "exit_blocked", "blocked_door"],
    "EQUIPMENT-notification": ["notification_appliance", "horn_strobe", "horn", "strobe", "speaker", "bell"],
    "EQUIPMENT-pull_station": ["pull_station", "pull_box", "manual_station"],
    "EQUIPMENT-fire_panel": ["fire_alarm_panel", "facp", "alarm_panel", "control_panel"],
}

# Classes that indicate specific violations
MISSING_CLASSES = {"empty_mount", "extinguisher_cabinet_empty", "bracket_empty"}
NONCOMPLIANT_CLASSES = {"non_compliant_tag", "noncompliant_tag", "yellow_tag", "red_tag"}
EXIT_DARK_CLASSES = {"exit_sign_dark", "exit_dark", "unlit_exit"}
SMOKE_MISSING_CLASSES = {"smoke_detector_missing", "detector_missing"}
BLOCKED_EXIT_CLASSES = {"blocked_exit", "exit_blocked"}

# All violation classes (for quick lookup)
ALL_VIOLATION_CLASSES = (MISSING_CLASSES | NONCOMPLIANT_CLASSES | EXIT_DARK_CLASSES |
                         SMOKE_MISSING_CLASSES | BLOCKED_EXIT_CLASSES)

FIRE_CLASS_LETTERS = {"a", "b", "c", "d", "e", "f", "k"}

def classify_service(class_name):
    """Determine which FireSafetyNet service a class belongs to."""
    name_lower = class_name.lower().strip().replace(" ", "_").replace("-", "_")
    # Check violation/condition classes first
    if name_lower in MISSING_CLASSES or "empty" in name_lower:
        return "VIOLATION-missing"
    if name_lower in NONCOMPLIANT_CLASSES:
        return "VIOLATION-noncompliant"
    if name_lower in EXIT_DARK_CLASSES:
        return "VIOLATION-exit_dark"
    if name_lower in SMOKE_MISSING_CLASSES:
        return "VIOLATION-smoke_missing"
    if name_lower in BLOCKED_EXIT_CLASSES:
        return "VIOLATION-blocked_exit"
    # Single-letter fire class symbols
    if name_lower in FIRE_CLASS_LETTERS:
        return "S6-fire_class"
    if name_lower.isdigit():
        return "S6-fire_class"
    for service, keywords in SERVICE_KEYWORDS.items():
        for kw in keywords:
            if kw in name_lower:
                return service
    return "unknown"

def boxes_overlap_or_near(bbox1, bbox2, margin=50):
    """Check if two bounding boxes overlap or are within `margin` pixels."""
    x1a, y1a, x2a, y2a = bbox1
    x1b, y1b, x2b, y2b = bbox2
    x1a -= margin; y1a -= margin; x2a += margin; y2a += margin
    return not (x2a < x1b or x2b < x1a or y2a < y1b or y2b < y1a)

# --- Load models from the repo ---
detect_model = None
segment_model = None

model_candidates = [
    ("best_detect.pt", "detect"),
    ("best_segment.pt", "segment"),
    ("best.pt", "detect"),
]

for model_file, expected_task in model_candidates:
    p = Path(model_file)
    if not p.exists():
        continue
    m = YOLO(str(p))
    t = m.overrides.get("task", "detect")
    if t == "segment" and segment_model is None:
        segment_model = m
        print(f"Segmentation model loaded: {p}")
        print(f"  Classes ({len(m.names)}):")
        for idx, cls in m.names.items():
            svc = classify_service(cls)
            print(f"    [{idx}] {cls}  -> {svc}")
        print()
    elif t != "segment" and detect_model is None:
        detect_model = m
        print(f"Detection model loaded: {p}")
        print(f"  Classes ({len(m.names)}):")
        for idx, cls in m.names.items():
            svc = classify_service(cls)
            print(f"    [{idx}] {cls}  -> {svc}")
        print()

if detect_model is None and segment_model is None:
    print("No model files found in repo. Falling back to manual upload...")
    from google.colab import files
    try:
        up = files.upload()
        for name in up.keys():
            p = Path(name)
            m = YOLO(str(p))
            t = m.overrides.get("task", "detect")
            if t == "segment" and segment_model is None:
                segment_model = m
                print(f"Segmentation model loaded: {p}")
            elif detect_model is None:
                detect_model = m
                print(f"Detection model loaded: {p}")
    except Exception:
        pass

if detect_model is None and segment_model is None:
    raise FileNotFoundError("No models found. Add best.pt, best_detect.pt, or best_segment.pt to the repo.")

# Build service map
detect_service_map = {}
if detect_model:
    for idx, cls in detect_model.names.items():
        detect_service_map[cls] = classify_service(cls)
seg_service_map = {}
if segment_model:
    for idx, cls in segment_model.names.items():
        seg_service_map[cls] = classify_service(cls)

# Summary
models_loaded = []
if detect_model: models_loaded.append("Detection")
if segment_model: models_loaded.append("Segmentation")
print(f"Models ready: {', '.join(models_loaded)}")

# Check what the model can detect
all_services = set(detect_service_map.values()) | set(seg_service_map.values())
detected_violations = [s for s in all_services if "VIOLATION" in s]
detected_equipment = [s for s in all_services if "EQUIPMENT" in s]

if detected_violations:
    print(f"\n  Violation classes active: {', '.join(detected_violations)}")
if detected_equipment:
    print(f"  Equipment classes active: {', '.join(detected_equipment)}")
if any("S5" in v for v in all_services):
    print("  S5 (inspection tags): will OCR detected tag regions for expiry dates")
if any("S6" in v for v in all_services):
    print("  S6 (fire class symbols): will identify fire class symbols")

# Show what's NOT yet trainable
needed = {
    "VIOLATION-missing": "empty_mount (extinguisher missing from bracket)",
    "VIOLATION-noncompliant": "non_compliant_tag (red/yellow tag on extinguisher)",
    "VIOLATION-exit_dark": "exit_sign_dark (exit sign not illuminated)",
    "VIOLATION-smoke_missing": "smoke_detector_missing (base without detector)",
    "VIOLATION-blocked_exit": "blocked_exit (exit blocked by objects)",
    "EQUIPMENT-notification": "notification_appliance (horn/strobe for height check)",
    "EQUIPMENT-pull_station": "pull_station (manual pull station for height check)",
    "EQUIPMENT-fire_panel": "fire_alarm_panel (FACP for status check)",
}
missing_services = [desc for svc, desc in needed.items() if svc not in all_services]
if missing_services:
    print(f"\n  Classes to add via feedback loop ({len(missing_services)} remaining):")
    for desc in missing_services:
        print(f"    - {desc}")

## 4. Load video from repo

Loads a video from the `videos/` directory in the repository.
Add your test videos to `flashinspector-ai/videos/` and push to GitHub.

If no videos are found in the repo, you can upload one manually.

In [ ]:
from pathlib import Path

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
videos_dir = Path("videos")

# Find all video files in the videos/ directory
video_files = sorted(
    [f for f in videos_dir.glob("*") if f.suffix.lower() in VIDEO_EXTENSIONS]
) if videos_dir.exists() else []

if video_files:
    print(f"Found {len(video_files)} video(s) in videos/ directory:\n")
    for i, v in enumerate(video_files):
        size_mb = v.stat().st_size / 1024 / 1024
        print(f"  [{i}] {v.name}  ({size_mb:.1f} MB)")

    # Use the first video by default, or change the index below
    selected = 0  # <-- change this index to pick a different video
    VIDEO_PATH = video_files[selected]
    print(f"\nUsing: {VIDEO_PATH} ({VIDEO_PATH.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    print("No videos found in videos/ directory. Falling back to manual upload...")
    from google.colab import files
    uploaded = files.upload()
    VIDEO_PATH = Path(list(uploaded.keys())[0])
    print(f"\nUploaded: {VIDEO_PATH} ({VIDEO_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

## 5. Run inference on video (both models)

Runs both detection and segmentation models on each frame and combines the annotations.

**Service-aware processing:**
- **S1** (equipment): fire extinguishers, blankets, smoke detectors, call points
- **S2** (marking): FSE marking signs
- **S3/S4** (condition): segmentation masks for extinguisher condition
- **S5** (inspection tags): detects tag region → **OCR reads the date** → checks if expired
- **S6** (fire class symbols): identifies fire classes on extinguisher labels

**Violation detection:**
- **EXTINGUISHER MISSING**: `empty_mount` with no nearby extinguisher → red flag
- **NON-COMPLIANT**: `non_compliant_tag` on extinguisher → orange flag
- **EXIT NOT ILLUMINATED**: `exit_sign_dark` detected → red flag
- **SMOKE DETECTOR MISSING**: `smoke_detector_missing` base → red flag
- **EXIT BLOCKED**: `blocked_exit` detected → red flag
- **Mounting height**: `notification_appliance`/`pull_station` position in frame → warning if in top 25%

In [ ]:
import cv2
import numpy as np
import re
import time
from datetime import datetime, date
from pathlib import Path

# ── Settings ──────────────────────────────────────────────────────────
CONFIDENCE = 0.15       # lower than default to catch small tags/symbols
FRAME_SKIP = 5          # process every Nth frame
OCR_CONF_MIN = 0.40     # only OCR extinguisher labels above this confidence
OCR_COOLDOWN = 10       # skip OCR for same-region detections within N frames
MISSING_MARGIN = 80     # pixels: how close an extinguisher must be to not flag missing
HEIGHT_WARN_ZONE = 0.25 # top 25% of frame → "TOO HIGH" warning for appliances

# ── Class name consolidation ──────────────────────────────────────────
CLASS_MAP = {
    "right exit": "emergency_exit", "left exit": "emergency_exit",
    "Right Exit": "emergency_exit", "Left Exit": "emergency_exit",
    "Straight Exit": "emergency_exit", "straight exit": "emergency_exit",
    "Left-Right Exit": "emergency_exit", "left-right exit": "emergency_exit",
    "emergency exit": "emergency_exit", "Emergency Exit": "emergency_exit",
    "fire-extinguisher": "fire_extinguisher",
    "fire extinguisher": "fire_extinguisher",
    "Fire_Extinguisher": "fire_extinguisher",
    "yellow tag": "yellow_tag",
    "red tag": "red_tag",
    "white tag": "white_tag",
    "Emergency-Light": "emergency_light",
    "emergency-light": "emergency_light",
    "Alarm": "alarm",
    "Alarm(Bell)": "alarm_bell",
    "Alarm(Lever)": "alarm_lever",
    "Alarm (Lever)": "alarm_lever",
    "Fire-Extinguisher": "fire_extinguisher",
}

def consolidate_class(name):
    return CLASS_MAP.get(name, name)

# ── Violation banner drawing ──────────────────────────────────────────
def draw_violation_banner(img, bbox, text, color=(0, 0, 255)):
    """Draw a colored violation banner above a bounding box."""
    mb = [int(x) for x in bbox]
    cv2.rectangle(img, (mb[0]-3, mb[1]-3), (mb[2]+3, mb[3]+3), color, 4)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
    by = max(mb[1] - 30, th + 10)
    cv2.rectangle(img, (mb[0], by - th - 8), (mb[0] + tw + 10, by + 4), color, -1)
    cv2.putText(img, text, (mb[0] + 5, by - 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)

# ── OCR setup for inspection tag reading (S5) ────────────────────────
import easyocr
ocr_reader = easyocr.Reader(["en"], gpu=True, verbose=False)
print("OCR reader loaded for inspection tag reading.\n")

def preprocess_for_ocr(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    if max(h, w) < 200:
        scale = 200 / max(h, w)
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 15, 4)
    return binary

def is_useful_text(text):
    t = text.strip()
    if len(t) < 2: return False
    alnum = sum(c.isalnum() for c in t)
    if alnum == 0: return False
    if alnum / len(t) < 0.5: return False
    if not re.search(r'[aeiouAEIOU0-9]', t) and len(t) > 3: return False
    return True

def bbox_key(bbox, grid=80):
    cx = int((bbox[0] + bbox[2]) / 2) // grid
    cy = int((bbox[1] + bbox[3]) / 2) // grid
    return (cx, cy)

def read_tag_region(frame, bbox, padding=5):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1, y1 = max(0, x1 - padding), max(0, y1 - padding)
    x2, y2 = min(w, x2 + padding), min(h, y2 + padding)
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0: return []
    processed = preprocess_for_ocr(crop)
    texts = ocr_reader.readtext(processed, detail=0, paragraph=False)
    return [t for t in texts if is_useful_text(t)]

def extract_label_region(frame, bbox):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    box_h, box_w = y2 - y1, x2 - x1
    crop = frame[y1+int(box_h*0.30):y1+int(box_h*0.80), x1+int(box_w*0.10):x2-int(box_w*0.10)]
    if crop.size == 0: return []
    processed = preprocess_for_ocr(crop)
    texts = ocr_reader.readtext(processed, detail=0, paragraph=False)
    return [t for t in texts if is_useful_text(t)]

def parse_dates(texts):
    found = []
    patterns = [
        (r'(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})', "dmy4"),
        (r'(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{2})', "dmy2"),
        (r'(\d{4})[/\-.](\d{1,2})[/\-.](\d{1,2})', "ymd"),
        (r'(\w+)\s+(\d{4})', "month_year"),
        (r'(\d{1,2})[/\-.](\d{4})', "my"),
    ]
    for text in texts:
        for pattern, fmt in patterns:
            m = re.search(pattern, text)
            if not m: continue
            try:
                if fmt == "dmy4":
                    d = datetime.strptime(f"{m.group(1)}/{m.group(2)}/{m.group(3)}", "%d/%m/%Y").date()
                elif fmt == "dmy2":
                    d = datetime.strptime(f"{m.group(1)}/{m.group(2)}/{m.group(3)}", "%d/%m/%y").date()
                elif fmt == "ymd":
                    d = datetime.strptime(f"{m.group(1)}/{m.group(2)}/{m.group(3)}", "%Y/%m/%d").date()
                elif fmt == "month_year":
                    d = None
                    for mfmt in ["%B %Y", "%b %Y"]:
                        try:
                            d = datetime.strptime(f"{m.group(1)} {m.group(2)}", mfmt).date()
                            break
                        except ValueError: continue
                    if d is None: continue
                elif fmt == "my":
                    d = datetime.strptime(f"{m.group(1)}/{m.group(2)}", "%m/%Y").date()
                found.append((d, text.strip()))
            except ValueError: continue
    return found

def check_expiry(dates):
    if not dates: return None, None, None
    today = date.today()
    latest = max(dates, key=lambda x: x[0])
    return latest[0] < today, latest[0], latest[1]

# ── Service-aware color scheme ────────────────────────────────────────
SERVICE_COLORS = {
    "S1-equipment": (0, 255, 0),             # green
    "S2-marking": (255, 165, 0),              # orange
    "S5-inspection_tag": (0, 255, 255),       # yellow
    "S6-fire_class": (255, 0, 255),           # magenta
    "VIOLATION-missing": (0, 0, 255),         # red
    "VIOLATION-noncompliant": (0, 100, 255),  # dark orange
    "VIOLATION-exit_dark": (0, 0, 200),       # dark red
    "VIOLATION-smoke_missing": (0, 0, 255),   # red
    "VIOLATION-blocked_exit": (0, 0, 255),    # red
    "EQUIPMENT-notification": (255, 200, 0),  # cyan-ish
    "EQUIPMENT-pull_station": (200, 0, 0),    # dark blue
    "EQUIPMENT-fire_panel": (128, 128, 0),    # teal
    "unknown": (200, 200, 200),               # gray
}

# ── Open video ────────────────────────────────────────────────────────
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video: {VIDEO_PATH.name}")
print(f"Resolution: {width}x{height}, FPS: {fps:.1f}, Frames: {total_frames}, Duration: {total_frames/fps:.1f}s")
active = []
if detect_model: active.append("Detection")
if segment_model: active.append("Segmentation")
print(f"Running: {' + '.join(active)}  (conf={CONFIDENCE})")
print(f"Violation detection: missing_margin={MISSING_MARGIN}px, height_zone={HEIGHT_WARN_ZONE}")
print(f"Processing every {FRAME_SKIP} frame(s)...\n")

# ── Output video ──────────────────────────────────────────────────────
out_dir = Path("inference_results")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / f"result_{VIDEO_PATH.stem}.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(str(out_path), fourcc, fps / FRAME_SKIP, (width, height))

# ── Trackers ──────────────────────────────────────────────────────────
detect_detections = []
segment_detections = []
tag_readings = []
expiry_results = []
symbol_detections = []
violations = {
    "missing": [],          # empty mount, no nearby extinguisher
    "non_compliant": [],    # non-compliant tag on extinguisher
    "exit_dark": [],        # exit sign not illuminated
    "smoke_missing": [],    # smoke detector missing from base
    "blocked_exit": [],     # exit blocked
    "height_warning": [],   # notification appliance or pull station too high
}
ocr_last_frame = {}
ocr_skipped = 0

frame_idx = 0
processed = 0
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret: break

    if frame_idx % FRAME_SKIP == 0:
        annotated = frame.copy()
        timestamp = round(frame_idx / fps, 2)

        # ── Segmentation model ────────────────────────────────────────
        if segment_model:
            seg_results = segment_model(frame, conf=CONFIDENCE, verbose=False)[0]
            annotated = seg_results.plot(img=annotated, boxes=False)
            for box in seg_results.boxes:
                raw_cls = seg_results.names[int(box.cls)]
                cls_name = consolidate_class(raw_cls)
                svc = seg_service_map.get(raw_cls, classify_service(raw_cls))
                segment_detections.append({
                    "frame": frame_idx, "timestamp_sec": timestamp,
                    "class": cls_name, "confidence": round(float(box.conf), 3),
                    "model": "segment", "service": svc,
                })

        # ── Detection model ───────────────────────────────────────────
        if detect_model:
            det_results = detect_model(frame, conf=CONFIDENCE, verbose=False)[0]
            frame_boxes = []  # (cls_name, svc, bbox_float, conf_val)

            for box in det_results.boxes:
                bbox_float = [float(x) for x in box.xyxy[0].tolist()]
                bbox = [int(x) for x in box.xyxy[0].tolist()]
                raw_cls = det_results.names[int(box.cls)]
                cls_name = consolidate_class(raw_cls)
                conf_val = float(box.conf)
                svc = detect_service_map.get(raw_cls, classify_service(raw_cls))

                frame_boxes.append((cls_name, svc, bbox_float, conf_val))
                color = SERVICE_COLORS.get(svc, SERVICE_COLORS["unknown"])

                # Draw bounding box
                cv2.rectangle(annotated, (bbox[0], bbox[1]), (bbox[2], bbox[3]), color, 2)
                label = f"{cls_name} {conf_val:.2f}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
                cv2.rectangle(annotated, (bbox[0], bbox[1]-th-6), (bbox[0]+tw, bbox[1]), color, -1)
                cv2.putText(annotated, label, (bbox[0], bbox[1]-4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 1, cv2.LINE_AA)

                detect_detections.append({
                    "frame": frame_idx, "timestamp_sec": timestamp,
                    "class": cls_name, "confidence": round(conf_val, 3),
                    "model": "detect", "bbox": bbox_float, "service": svc,
                })

                # ── S5: Inspection tag OCR ────────────────────────────
                if "S5" in svc:
                    bk = bbox_key(bbox_float)
                    if frame_idx - ocr_last_frame.get(("s5", bk), -999) >= OCR_COOLDOWN:
                        ocr_last_frame[("s5", bk)] = frame_idx
                        texts = read_tag_region(frame, bbox_float)
                        if texts:
                            tag_readings.append({"frame": frame_idx, "timestamp_sec": timestamp, "texts": texts, "bbox": bbox_float})
                            dates = parse_dates(texts)
                            is_expired, exp_date, raw = check_expiry(dates)
                            if is_expired is not None:
                                expiry_results.append({"frame": frame_idx, "timestamp_sec": timestamp, "expired": is_expired, "date": exp_date, "raw": raw})
                                sc = (0,0,255) if is_expired else (0,200,0)
                                st = f"EXPIRED {exp_date}" if is_expired else f"Valid until {exp_date}"
                                cv2.putText(annotated, st, (bbox[0], bbox[3]+20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, sc, 2, cv2.LINE_AA)
                            else:
                                y_off = bbox[3]+18
                                for txt in texts[:3]:
                                    cv2.putText(annotated, txt[:50], (bbox[0], y_off), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,255,255), 1, cv2.LINE_AA)
                                    y_off += 16
                    else: ocr_skipped += 1

                # ── S6: Fire class symbol ─────────────────────────────
                if "S6" in svc:
                    symbol_detections.append({"frame": frame_idx, "timestamp_sec": timestamp, "class": cls_name, "confidence": round(conf_val, 3)})
                    bk = bbox_key(bbox_float)
                    if frame_idx - ocr_last_frame.get(("s6", bk), -999) >= OCR_COOLDOWN:
                        ocr_last_frame[("s6", bk)] = frame_idx
                        sym_texts = read_tag_region(frame, bbox_float)
                        if sym_texts:
                            y_off = bbox[3]+18
                            for txt in sym_texts[:2]:
                                cv2.putText(annotated, txt[:40], (bbox[0], y_off), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,0,255), 1, cv2.LINE_AA)
                                y_off += 16
                    else: ocr_skipped += 1

                # ── S1: OCR extinguisher labels ───────────────────────
                if "extinguisher" in cls_name.lower() and "S1" in svc and conf_val >= OCR_CONF_MIN:
                    bk = bbox_key(bbox_float)
                    if frame_idx - ocr_last_frame.get(("ext", bk), -999) >= OCR_COOLDOWN:
                        ocr_last_frame[("ext", bk)] = frame_idx
                        texts = extract_label_region(frame, bbox_float)
                        if texts:
                            tag_readings.append({"frame": frame_idx, "timestamp_sec": timestamp, "texts": texts, "bbox": bbox_float, "source": "extinguisher_label"})
                            dates = parse_dates(texts)
                            is_expired, exp_date, raw = check_expiry(dates)
                            if is_expired is not None:
                                sc = (0,0,255) if is_expired else (0,200,0)
                                st = f"EXPIRED {exp_date}" if is_expired else f"Valid {exp_date}"
                                cv2.putText(annotated, st, (bbox[0], bbox[3]+20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, sc, 2, cv2.LINE_AA)
                                expiry_results.append({"frame": frame_idx, "timestamp_sec": timestamp, "expired": is_expired, "date": exp_date, "raw": raw})
                    else: ocr_skipped += 1

                # ── VIOLATION: Non-compliant tag ──────────────────────
                if "VIOLATION-noncompliant" in svc or cls_name.lower() in NONCOMPLIANT_CLASSES:
                    violations["non_compliant"].append({
                        "frame": frame_idx, "timestamp_sec": timestamp,
                        "class": cls_name, "confidence": round(conf_val, 3), "bbox": bbox_float,
                    })
                    draw_violation_banner(annotated, bbox_float, "NON-COMPLIANT TAG", (0, 100, 255))

                # ── VIOLATION: Exit sign not illuminated ──────────────
                if "VIOLATION-exit_dark" in svc or cls_name.lower() in EXIT_DARK_CLASSES:
                    violations["exit_dark"].append({
                        "frame": frame_idx, "timestamp_sec": timestamp,
                        "class": cls_name, "confidence": round(conf_val, 3), "bbox": bbox_float,
                    })
                    draw_violation_banner(annotated, bbox_float, "EXIT SIGN NOT ILLUMINATED", (0, 0, 200))

                # ── VIOLATION: Smoke detector missing from base ───────
                if "VIOLATION-smoke_missing" in svc or cls_name.lower() in SMOKE_MISSING_CLASSES:
                    violations["smoke_missing"].append({
                        "frame": frame_idx, "timestamp_sec": timestamp,
                        "class": cls_name, "confidence": round(conf_val, 3), "bbox": bbox_float,
                    })
                    draw_violation_banner(annotated, bbox_float, "SMOKE DETECTOR MISSING", (0, 0, 255))

                # ── VIOLATION: Blocked exit ───────────────────────────
                if "VIOLATION-blocked_exit" in svc or cls_name.lower() in BLOCKED_EXIT_CLASSES:
                    violations["blocked_exit"].append({
                        "frame": frame_idx, "timestamp_sec": timestamp,
                        "class": cls_name, "confidence": round(conf_val, 3), "bbox": bbox_float,
                    })
                    draw_violation_banner(annotated, bbox_float, "EXIT BLOCKED", (0, 0, 255))

                # ── HEIGHT CHECK: Notification appliance / pull station
                if "EQUIPMENT-notification" in svc or "EQUIPMENT-pull_station" in svc:
                    # If the center of the detection is in the top portion of the frame,
                    # flag as potentially mounted too high
                    center_y = (bbox_float[1] + bbox_float[3]) / 2
                    if center_y < height * HEIGHT_WARN_ZONE:
                        equip_type = "APPLIANCE" if "notification" in svc else "PULL STATION"
                        violations["height_warning"].append({
                            "frame": frame_idx, "timestamp_sec": timestamp,
                            "class": cls_name, "confidence": round(conf_val, 3),
                            "bbox": bbox_float, "type": equip_type,
                        })
                        draw_violation_banner(annotated, bbox_float, f"{equip_type} TOO HIGH?", (0, 180, 255))

            # ── VIOLATION: Extinguisher missing ───────────────────────
            extinguisher_bboxes = [b[2] for b in frame_boxes if "extinguisher" in b[0].lower() and "VIOLATION" not in b[1]]
            empty_mounts = [(b[0], b[2], b[3]) for b in frame_boxes if "VIOLATION-missing" in b[1] or b[0].lower() in MISSING_CLASSES]

            for mount_cls, mount_bbox, mount_conf in empty_mounts:
                has_nearby = any(
                    boxes_overlap_or_near(mount_bbox, ext_bb, margin=MISSING_MARGIN)
                    for ext_bb in extinguisher_bboxes
                )
                if not has_nearby:
                    violations["missing"].append({
                        "frame": frame_idx, "timestamp_sec": timestamp,
                        "class": mount_cls, "confidence": round(mount_conf, 3), "bbox": mount_bbox,
                    })
                    draw_violation_banner(annotated, mount_bbox, "EXTINGUISHER MISSING", (0, 0, 255))

        out_writer.write(annotated)
        processed += 1

        if processed % 50 == 0:
            elapsed = time.time() - start_time
            pct = frame_idx / total_frames * 100
            print(f"  {pct:.0f}% — {processed} frames ({processed/elapsed:.1f} fps)")

    frame_idx += 1

cap.release()
out_writer.release()

elapsed = time.time() - start_time
all_detections = detect_detections + segment_detections
total_violations = sum(len(v) for v in violations.values())

print(f"\nDone! {processed} frames in {elapsed:.1f}s ({processed/max(elapsed,0.01):.1f} fps)")
print(f"Annotated video: {out_path}")
if tag_readings:
    print(f"OCR: {len(tag_readings)} tag/label reads ({ocr_skipped} skipped)")
if expiry_results:
    n_exp = sum(1 for e in expiry_results if e["expired"])
    print(f"Expiry: {n_exp} expired, {len(expiry_results)-n_exp} valid")
if total_violations:
    print(f"\nVIOLATIONS FOUND: {total_violations} total")
    for vtype, vlist in violations.items():
        if vlist:
            print(f"  {vtype}: {len(vlist)} in {len(set(v['frame'] for v in vlist))} frames")
else:
    print("\nNo violations detected. Train new classes via the feedback loop:")
    print("  Annotate empty_mount, non_compliant_tag, exit_sign_dark, etc.")

## 6. Results — service-by-service breakdown + compliance report

Shows detections grouped by FireSafetyNet service, OCR readings from inspection tags, expiry status, and condition check results.

In [ ]:
from collections import Counter

def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

# ── Detection model by service ────────────────────────────────────────
if detect_detections:
    section("DETECTION MODEL — by service")
    by_svc = {}
    for d in detect_detections:
        svc = d.get("service", "unknown")
        by_svc.setdefault(svc, []).append(d)
    for svc in sorted(by_svc):
        dets = by_svc[svc]
        counts = Counter(d["class"] for d in dets)
        print(f"\n  [{svc}] — {len(dets)} detections")
        for cls, count in counts.most_common():
            confs = [d["confidence"] for d in dets if d["class"] == cls]
            print(f"    {cls}: {count} (avg conf: {sum(confs)/len(confs):.2f})")

# ── Segmentation model ───────────────────────────────────────────────
if segment_detections:
    section("SEGMENTATION MODEL — mask detections")
    counts = Counter(d["class"] for d in segment_detections)
    print(f"  Total: {len(segment_detections)} mask detections\n")
    for cls, count in counts.most_common():
        confs = [d["confidence"] for d in segment_detections if d["class"] == cls]
        print(f"    {cls}: {count} (avg conf: {sum(confs)/len(confs):.2f})")

# ── ALL VIOLATIONS ───────────────────────────────────────────────────
VIOLATION_LABELS = {
    "missing": ("EXTINGUISHER MISSING", "Empty mount/cabinet with no nearby extinguisher"),
    "non_compliant": ("NON-COMPLIANT TAG", "Red/yellow tag indicating failed inspection"),
    "exit_dark": ("EXIT SIGN NOT ILLUMINATED", "Dark or non-functioning exit sign"),
    "smoke_missing": ("SMOKE DETECTOR MISSING", "Detector base on ceiling without unit"),
    "blocked_exit": ("EXIT BLOCKED", "Emergency exit obstructed by objects"),
    "height_warning": ("MOUNTED TOO HIGH", "Notification appliance or pull station above normal height"),
}

total_violations = sum(len(v) for v in violations.values())
section(f"VIOLATIONS REPORT — {total_violations} total")

if total_violations:
    for vtype, vlist in violations.items():
        if not vlist:
            continue
        label, desc = VIOLATION_LABELS.get(vtype, (vtype, ""))
        v_frames = len(set(v["frame"] for v in vlist))
        print(f"\n  [{label}] — {len(vlist)} detections across {v_frames} frames")
        print(f"  {desc}")

        # Show unique timestamps
        seen = set()
        for v in vlist:
            t = v["timestamp_sec"]
            if t not in seen:
                seen.add(t)
                extra = f" ({v.get('type', '')})" if v.get("type") else ""
                print(f"    @ {t}s — {v['class']} (conf={v['confidence']:.2f}){extra}")
                if len(seen) >= 15:
                    remaining = len(set(vi["timestamp_sec"] for vi in vlist)) - 15
                    if remaining > 0:
                        print(f"    ... and {remaining} more timestamps")
                    break
else:
    print("  No violations detected.")
    print("\n  To detect violations, train these classes via the feedback loop:")
    print("    - empty_mount / extinguisher_cabinet_empty -> EXTINGUISHER MISSING")
    print("    - non_compliant_tag -> NON-COMPLIANT (most common in your videos)")
    print("    - exit_sign_dark -> EXIT SIGN NOT ILLUMINATED")
    print("    - smoke_detector_missing -> SMOKE DETECTOR MISSING")
    print("    - blocked_exit -> EXIT BLOCKED")
    print("    - notification_appliance -> height check")
    print("    - pull_station -> height check")

# ── S5: Inspection tag OCR readings ──────────────────────────────────
section("S5 — INSPECTION TAG READINGS (OCR)")
if tag_readings:
    text_counts = Counter()
    text_first_seen = {}
    for tr in tag_readings:
        src = tr.get("source", "inspection_tag")
        for txt in tr["texts"]:
            norm = txt.strip()
            if norm:
                text_counts[(norm, src)] += 1
                if (norm, src) not in text_first_seen:
                    text_first_seen[(norm, src)] = tr["timestamp_sec"]
    print(f"  Read text from {len(tag_readings)} tag/label regions")
    print(f"  Unique text strings: {len(text_counts)}\n")
    for (txt, src), count in text_counts.most_common(30):
        first_t = text_first_seen[(txt, src)]
        print(f"    {count:>3}x  ({src}) \"{txt}\"  [first @ {first_t}s]")
else:
    print("  No inspection tag text detected.")

# ── S5: Expiry date check ────────────────────────────────────────────
section("S5 — EXPIRY DATE CHECK")
if expiry_results:
    from datetime import date
    today = date.today()
    for e in expiry_results:
        icon = "[X]" if e["expired"] else "[OK]"
        status = "EXPIRED" if e["expired"] else "VALID"
        print(f"  {icon} {status}: {e['date']} (from \"{e['raw']}\") @ {e['timestamp_sec']}s")
    n_exp = sum(1 for e in expiry_results if e["expired"])
    print(f"\n  Summary: {n_exp} EXPIRED, {len(expiry_results)-n_exp} VALID (today: {today})")
else:
    print("  No dates found on tags/labels.")

# ── S6: Fire class symbols ───────────────────────────────────────────
if symbol_detections:
    section("S6 — FIRE CLASS SYMBOLS")
    sym_counts = Counter(d["class"] for d in symbol_detections)
    for cls, count in sym_counts.most_common():
        print(f"    {cls}: detected {count} times")

# ── Combined totals ──────────────────────────────────────────────────
section("COMBINED TOTALS")
print(f"  Detection model: {len(detect_detections)} detections")
print(f"  Segmentation model: {len(segment_detections)} mask detections")
print(f"  Violations: {total_violations}")
for vtype, vlist in violations.items():
    if vlist:
        label = VIOLATION_LABELS.get(vtype, (vtype,))[0]
        print(f"    - {label}: {len(vlist)}")
print(f"  OCR tag/label reads: {len(tag_readings)}")
print(f"  Expiry dates found: {len(expiry_results)}")
print(f"  Fire class symbols: {len(symbol_detections)}")

if not detect_detections and not segment_detections:
    print(f"\n  No detections. Try lowering CONFIDENCE (currently {CONFIDENCE}).")

## 7. Preview a sample frame

In [ ]:
import cv2
from IPython.display import display, Image as IPImage
from pathlib import Path
import tempfile

# Open the annotated video and grab a frame from the middle
cap = cv2.VideoCapture(str(out_path))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)  # jump to middle
ret, frame = cap.read()
cap.release()

if ret:
    tmp = Path(tempfile.mktemp(suffix=".jpg"))
    cv2.imwrite(str(tmp), frame)
    display(IPImage(str(tmp), width=800))
    print(f"Showing frame {total // 2} of {total}")
else:
    print("Could not read a frame from the output video.")

## 8. Download the annotated video

In [ ]:
from google.colab import files
from pathlib import Path

if Path(out_path).exists():
    print(f"Downloading: {out_path} ({Path(out_path).stat().st_size / 1024 / 1024:.1f} MB)")
    files.download(str(out_path))
else:
    print("Output video not found. Run the inference cell above first.")

## 9. Roboflow Cloud Inference — Empty Mount Detection

Run your **custom Roboflow model** (`my-first-project-nqfzv` v3) on a video using the
Roboflow HTTP API. Sends every Nth frame for inference and draws bounding boxes.

**Requirements:** `inference-sdk` (installed below). Upload a video or use the one
from Section 3.

**403 Forbidden?** The `model_id` must be `workspace/project_slug/version` (e.g. `patyas-workspace/my-first-project-nqfzv/3`).
Using only `project/version` breaks the API URL.

In [ ]:
!pip install -q inference-sdk

import cv2
import numpy as np
import base64
import os
from inference_sdk import InferenceHTTPClient

# ── Configuration ─────────────────────────────────────────────────────
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", "YOUR_API_KEY")  # set in .env or paste here
MODEL_ID = "patyas-workspace/my-first-project-nqfzv/3"  # workspace/project/version (required; omitting workspace causes 403)
VIDEO_INPUT = str(VIDEO_PATH)  # reuse video from Section 3
VIDEO_OUTPUT_PATH = "roboflow_output.mp4"
FRAME_SKIP = 5  # process every Nth frame (lower = slower but more thorough)
CONFIDENCE = 0.25  # minimum confidence threshold

client = InferenceHTTPClient(
    api_url="https://detect.roboflow.com",
    api_key=ROBOFLOW_API_KEY,
)

# ── Process video frame by frame ─────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_INPUT)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(VIDEO_OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
frame_idx = 0
det_count = 0
last_boxes = []  # reuse detections for skipped frames

print(f"Processing {total} frames ({total/fps:.1f}s video) — inferring every {FRAME_SKIP} frames")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % FRAME_SKIP == 0:
        # Send frame to Roboflow
        _, buf = cv2.imencode(".jpg", frame)
        img_b64 = base64.b64encode(buf.tobytes()).decode("utf-8")
        result = client.infer(img_b64, model_id=MODEL_ID)
        last_boxes = [
            p for p in result.get("predictions", [])
            if p["confidence"] >= CONFIDENCE
        ]
        if last_boxes:
            det_count += len(last_boxes)
        print(f"\rFrame {frame_idx}/{total} — {len(last_boxes)} detections", end="")

    # Draw boxes on every frame (reusing last inference result)
    for p in last_boxes:
        x1 = int(p["x"] - p["width"] / 2)
        y1 = int(p["y"] - p["height"] / 2)
        x2 = int(p["x"] + p["width"] / 2)
        y2 = int(p["y"] + p["height"] / 2)
        label = f'{p["class"]} {p["confidence"]:.0%}'
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(frame, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print(f"\n\nDone! {frame_idx} frames → {VIDEO_OUTPUT_PATH}")
print(f"Total detections across inferred frames: {det_count}")
